# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not IN_COLAB and os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import sklearn  # noqa: E402

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 120)

CSV_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV_PATH), "starter CSV not found -- are you at the repo root?"
df = pd.read_csv(CSV_PATH)
os.makedirs("work/outputs", exist_ok=True)

print("pandas", pd.__version__, "| numpy", np.__version__, "| sklearn", sklearn.__version__)
print("seed", SEED, "| loaded", df.shape)

# --- The label, and the columns that ARE the label -------------------------
# trend_pct is not a feature. It is the label written as a number:
#   trend_pct == (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100
# Measured below, not assumed.
prev_pos = df[df["impressions_prev_30d"] > 0]
recomputed = (
    (prev_pos["impressions_last_30d"] - prev_pos["impressions_prev_30d"])
    / prev_pos["impressions_prev_30d"] * 100
)
print("\n--- is trend_pct derived from the impression columns? ---")
print(f"correlation with recomputed value : {prev_pos['trend_pct'].corr(recomputed):.6f}")
print(f"max absolute deviation            : {(prev_pos['trend_pct'] - recomputed).abs().max():.4f}")

y_all = df["trend_direction"].str.lower().eq("down")
print(f"\nlabel = trend_direction == 'down'  ({y_all.sum():,} of {len(df):,} rows)")
print(f"agrees with trend_pct < 0 on     : {(y_all[prev_pos.index] == (prev_pos['trend_pct'] < 0)).mean() * 100:.1f}% of eligible rows")

LEAK_COLS = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
print("\nnever fitted on:", LEAK_COLS)


pandas 3.0.5 | numpy 2.5.1 | sklearn 1.9.0
seed 42 | loaded (30000, 44)

--- is trend_pct derived from the impression columns? ---
correlation with recomputed value : 1.000000
max absolute deviation            : 0.0500

label = trend_direction == 'down'  (16,262 of 30,000 rows)
agrees with trend_pct < 0 on     : 87.0% of eligible rows

never fitted on: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
from sklearn.model_selection import GroupKFold

# --- Population: rows where decline is arithmetically possible -------------
# Week-4 scored every ranked row. 2,191 of those have impressions_prev_30d == 0,
# and impressions cannot fall below zero, so their declining rate is exactly 0.000.
# They are not hard cases the model gets right -- they are rows that could only
# ever be one thing. Dropped here, and the Week-4 rule is re-scored on what's left
# so the comparison is the same rows, not just the same metric.

ranked = df["avg_position"] > 0
can_decline = df["impressions_prev_30d"] > 0

impossible = df[ranked & ~can_decline]
print(f"ranked rows                       : {ranked.sum():,}")
print(f"  ...of which cannot decline      : {len(impossible):,}")
print(f"  their declining rate            : {impossible['trend_direction'].str.lower().eq('down').mean():.3f}")

pop = df[ranked & can_decline].reset_index(drop=True).copy()
y = pop["trend_direction"].str.lower().eq("down").to_numpy()
groups = pop["client_id"].to_numpy()

print(f"\nscored population                 : {len(pop):,}")
print(f"base rate, Week-4 population      : {df.loc[ranked, 'trend_direction'].str.lower().eq('down').mean():.4f}")
print(f"base rate, this population        : {y.mean():.4f}")

# --- Re-score the Week-4 rule on this population ---------------------------
tier_stats = df[ranked].groupby("position_tier").agg(
    ti=("impressions_90d", "sum"), tc=("clicks_90d", "sum")
)
TIER_CTR = tier_stats["tc"] / tier_stats["ti"]
EXPECTED_CLICKS_FLOOR = 5.0
IN_SCOPE_TIERS = ["page_1", "striking", "page_3_5", "top_3"]

pop["tier_expected_ctr"] = pop["position_tier"].map(TIER_CTR)
pop["expected_clicks"] = pop["impressions_90d"] * pop["tier_expected_ctr"]
pop["missed_clicks"] = pop["expected_clicks"] - pop["clicks_90d"]
_scorable = (pop["expected_clicks"] >= EXPECTED_CLICKS_FLOOR) & pop["position_tier"].isin(IN_SCOPE_TIERS)
_fires = _scorable & (pop["missed_clicks"] > 0)
pop["baseline_score"] = np.where(_fires, pop["missed_clicks"] / pop["expected_clicks"], 0.0)

print(f"\nrule fires on                     : {_fires.sum():,} rows ({_fires.mean() * 100:.1f}%)")
print(f"rows tied at the capped 1.0       : {(pop['baseline_score'] == 1.0).sum():,}")

# --- Grouped folds: a client is never split across train and test ----------
# Declining rate runs from 0.199 to 0.960 across clients, and one client is 26%
# of the population. A random split would let the model learn client identity.
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(pop, y, groups))

print(f"\n--- {N_FOLDS} grouped folds (split on client_id) ---")
rows = []
for i, (tr, te) in enumerate(folds):
    rows.append({
        "fold": i,
        "test_rows": len(te),
        "test_clients": pop.iloc[te]["client_id"].nunique(),
        "test_base_rate": round(y[te].mean(), 3),
        "leaked_clients": len(set(groups[tr]) & set(groups[te])),
    })
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nbase rate spread across folds     : {min(r['test_base_rate'] for r in rows):.3f} to "
      f"{max(r['test_base_rate'] for r in rows):.3f}")
print(f"any client in both train and test : {any(r['leaked_clients'] for r in rows)}")


ranked rows                       : 28,795
  ...of which cannot decline      : 2,191
  their declining rate            : 0.000

scored population                 : 26,604
base rate, Week-4 population      : 0.5645
base rate, this population        : 0.6110

rule fires on                     : 7,126 rows (26.8%)
rows tied at the capped 1.0       : 403



--- 5 grouped folds (split on client_id) ---
 fold  test_rows  test_clients  test_base_rate  leaked_clients
    0       6981             1           0.492               0
    1       4905             7           0.637               0
    2       4906             8           0.553               0
    3       4905             7           0.767               0
    4       4907             8           0.657               0

base rate spread across folds     : 0.492 to 0.767
any client in both train and test : False


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# --- Two feature sets ------------------------------------------------------
# permissive: everything that is not the label and not an identifier.
# strict    : also drops the four columns measured inside the label window.
#             They are not the label, but a decision made on 1 April could not
#             have read them. The gap between the two is the cost of that rule.

WINDOW_COLS = ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]
DROP_ALWAYS = (
    ["content_id", "client_id"] + LEAK_COLS
    + ["tier_expected_ctr", "expected_clicks", "missed_clicks", "baseline_score"]
)

permissive = [c for c in pop.columns if c not in DROP_ALWAYS]
strict = [c for c in permissive if c not in WINDOW_COLS]
FEATURE_SETS = {"permissive": permissive, "strict": strict}

for name, cols in FEATURE_SETS.items():
    print(f"{name:11s}: {len(cols)} columns")
print(f"\nstrict drops: {WINDOW_COLS}")


def build_pipe(model, cols):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(pop[c])]
    cat = [c for c in cols if c not in num]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    return Pipeline([("pre", pre), ("model", model)])


MODELS = {
    "LogReg": lambda: LogisticRegression(max_iter=2000, random_state=SEED),
    "RandomForest": lambda: RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED
    ),
}
KS = [10, 50, 100, 500]


def precision_at_k(scores, truth, k):
    order = np.argsort(-scores, kind="stable")
    return truth[order[:k]].mean()


# --- Score every fold: baseline first, then each model on each feature set --
records = []
oof = {}

for fi, (tr, te) in enumerate(folds):
    y_te = y[te]
    records.append({
        "arm": "baseline (Week-4 rule)", "features": "--", "fold": fi,
        "base_rate": y_te.mean(),
        "roc_auc": roc_auc_score(y_te, pop.iloc[te]["baseline_score"].to_numpy()),
        **{f"p@{k}": precision_at_k(pop.iloc[te]["baseline_score"].to_numpy(), y_te, k) for k in KS},
    })

    for set_name, cols in FEATURE_SETS.items():
        X = pop[cols]
        for model_name, make in MODELS.items():
            pipe = build_pipe(make(), cols)
            pipe.fit(X.iloc[tr], y[tr])
            p = pipe.predict_proba(X.iloc[te])[:, 1]
            oof.setdefault((model_name, set_name), np.zeros(len(pop)))[te] = p
            records.append({
                "arm": model_name, "features": set_name, "fold": fi,
                "base_rate": y_te.mean(),
                "roc_auc": roc_auc_score(y_te, p),
                **{f"p@{k}": precision_at_k(p, y_te, k) for k in KS},
            })
    print(f"fold {fi} done")

per_fold = pd.DataFrame(records)

# --- The comparison table: same rows, same folds, same metrics -------------
metric_cols = ["base_rate", "roc_auc"] + [f"p@{k}" for k in KS]
summary = per_fold.groupby(["arm", "features"], sort=False)[metric_cols].mean().round(3)
spread = per_fold.groupby(["arm", "features"], sort=False)["roc_auc"].std().round(3)
summary["auc_sd"] = spread

print("\n--- model vs baseline: mean across 5 grouped folds ---")
print(summary.to_string())
print("\n--- ROC AUC per fold (fold 0 is a single client) ---")
print(per_fold.pivot_table(index=["arm", "features"], columns="fold", values="roc_auc", sort=False).round(3).to_string())

per_fold.to_csv("work/outputs/w05_model_vs_baseline.csv", index=False)
print("\nwrote work/outputs/w05_model_vs_baseline.csv")


permissive : 38 columns
strict     : 34 columns

strict drops: ['clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']


fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done

--- model vs baseline: mean across 5 grouped folds ---
                                   base_rate  roc_auc  p@10   p@50  p@100  p@500  auc_sd
arm                    features                                                         
baseline (Week-4 rule) --              0.621    0.532  0.76  0.808  0.782  0.746   0.032
LogReg                 permissive      0.621    0.625  0.74  0.796  0.760  0.740   0.042
RandomForest           permissive      0.621    0.645  0.92  0.844  0.842  0.787   0.027
LogReg                 strict          0.621    0.586  0.72  0.724  0.706  0.704   0.044
RandomForest           strict          0.621    0.611  0.84  0.868  0.812  0.771   0.038

--- ROC AUC per fold (fold 0 is a single client) ---
fold                                   0      1      2      3      4
arm                    features                                     
baseline (Week-4 rule) --          0.565  0.552  0.491  0.505  0.547
LogReg                 permissive  0.662  0.642 

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
from sklearn.inspection import permutation_importance

# --- What does it lean on? -------------------------------------------------
# Permutation importance on one fold's held-out rows, measured on ROC AUC.
# Refit here so this cell stands alone.
tr, te = folds[0]
cols = FEATURE_SETS["permissive"]
rf = build_pipe(MODELS["RandomForest"](), cols).fit(pop[cols].iloc[tr], y[tr])

imp = permutation_importance(
    rf, pop[cols].iloc[te], y[te], scoring="roc_auc",
    n_repeats=5, random_state=SEED, n_jobs=-1,
)
top = (
    pd.DataFrame({"feature": cols, "auc_drop": imp.importances_mean, "sd": imp.importances_std})
    .sort_values("auc_drop", ascending=False)
    .head(12)
    .round(4)
)
print("--- permutation importance, fold 0 held-out rows (drop in ROC AUC) ---")
print(top.to_string(index=False))

# --- Where is it wrong? ----------------------------------------------------
pred = oof[("RandomForest", "permissive")]
err = pop[["client_id", "position_tier", "impressions_90d", "clicks_90d", "baseline_score"]].copy()
err["truth"] = y
err["p_decline"] = pred
err["called_decline"] = pred >= 0.5
err["correct"] = err["called_decline"] == err["truth"]

print(f"\noverall out-of-fold accuracy at 0.50: {err['correct'].mean():.3f}  (base rate {y.mean():.3f})")

print("\n--- accuracy by position tier ---")
print(err.groupby("position_tier").agg(
    n=("correct", "size"), base_rate=("truth", "mean"), accuracy=("correct", "mean")
).round(3).sort_values("n", ascending=False).to_string())

print("\n--- the five clients it reads worst (n >= 100) ---")
byc = err.groupby("client_id").agg(
    n=("correct", "size"), base_rate=("truth", "mean"),
    accuracy=("correct", "mean"), mean_p=("p_decline", "mean"),
).round(3)
print(byc[byc["n"] >= 100].sort_values("accuracy").head(5).to_string())

print("\n--- confident and wrong, both directions ---")
for label, sub in [
    ("said decline, held steady", err[(~err["truth"]) & (err["p_decline"] >= 0.5)]),
    ("said steady, declined", err[(err["truth"]) & (err["p_decline"] < 0.5)]),
]:
    print(f"\n{label}: {len(sub):,} rows")
    worst = sub.reindex(
        (sub["p_decline"] - 0.5).abs().sort_values(ascending=False).index
    ).head(3)
    print(worst[["position_tier", "impressions_90d", "clicks_90d", "baseline_score", "p_decline"]].round(3).to_string(index=False))

# --- Does the model beat the rule where the rule is silent? ---------------
silent = err["baseline_score"] == 0.0
print(f"\n--- rows the Week-4 rule scores 0.0 (no opinion): {silent.sum():,} ---")
print(f"their true declining rate        : {err.loc[silent, 'truth'].mean():.3f}")
print(f"model ROC AUC on those rows only : {roc_auc_score(err.loc[silent, 'truth'], err.loc[silent, 'p_decline']):.3f}")


--- permutation importance, fold 0 held-out rows (drop in ROC AUC) ---
              feature  auc_drop     sd
         avg_position    0.0443 0.0016
      clicks_last_30d    0.0355 0.0021
        position_tier    0.0221 0.0010
     content_age_days    0.0066 0.0016
      impressions_90d    0.0066 0.0007
             age_tier    0.0056 0.0011
    sessions_last_30d    0.0049 0.0015
    sessions_prev_30d    0.0039 0.0006
       age_tier_order    0.0028 0.0009
      impression_tier    0.0027 0.0005
days_with_impressions    0.0016 0.0008
        pageviews_90d    0.0013 0.0002

overall out-of-fold accuracy at 0.50: 0.657  (base rate 0.611)

--- accuracy by position tier ---
                   n  base_rate  accuracy
position_tier                            
page_1         10734      0.627     0.683
striking        6989      0.637     0.643
page_3_5        6961      0.584     0.621
deep            1136      0.400     0.667
top_3            784      0.703     0.744

--- the five clients it read

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.